# 02 — Replication: CSA-Optimized Lightweight ML Classifiers

This notebook replicates the 5 classifiers from Samantaray et al. (ISACC 2025):
- KNN-CSA, LR-CSA, RF-CSA, DT-CSA, AB-CSA
- Uses Crow Search Algorithm (CSA) for hyperparameter optimization
- Generates classification reports and confusion matrices
- Compares results to the original paper's Tables II–VII

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.preprocessing import load_dataset, preprocess
from src.classifiers import get_classifier_config, CLASSIFIER_CONFIGS
from src.csa_optimizer import optimize_classifier

sns.set_style("whitegrid")
RANDOM_STATE = 42
CLASS_NAMES = ["Normal", "TCP-SYN", "PortScan", "Overflow", "Blackhole", "Diversion"]

In [2]:
# Load and preprocess
df = load_dataset("../data/UNR-IDD.csv")
X_train, X_test, y_train, y_test, scaler, feature_names = preprocess(df)

Dataset loaded: 37410 samples
Class distribution:
Label
Blackhole    8420
Diversion    5615
Normal       3773
Overflow     1022
PortScan     9499
TCP-SYN      9081
Name: count, dtype: int64
Features: 26
Train: 29928 samples | Test: 7482 samples


In [3]:
# Train all 5 CSA-optimized classifiers
trained_models = {}
results = {}

for name in ["RF", "DT", "AB"]:
    print(f"\n{'='*50}")
    print(f"Optimizing {name} with CSA...")
    print(f"{'='*50}")

    config = get_classifier_config(name)
    best_params, best_acc = optimize_classifier(
        config["class"], config["param_grid"],
        X_train, y_train, pop_size=30, epoch=50, cv=3
    )

    print(f"Best params: {best_params}")
    print(f"Best CV accuracy: {best_acc:.4f}")

    # Train final model with best params
    model = config["class"](**best_params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)

    trained_models[name] = model
    results[name] = {"accuracy": acc, "report": report, "params": best_params}

    print(f"Test accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

    # Save model
    os.makedirs("../results", exist_ok=True)
    joblib.dump(model, f"../results/{name}_CSA_model.pkl")
    print(f"Model saved to ../results/{name}_CSA_model.pkl")

2026/04/07 06:41:54 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: OriginalCSA(epoch=50, pop_size=30, p_a=0.3)



Optimizing RF with CSA...


2026/04/07 06:45:54 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 1, Current best: 0.9382183908045977, Global best: 0.9382183908045977, Runtime: 130.56383 seconds
2026/04/07 06:48:17 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 2, Current best: 0.9382183908045977, Global best: 0.9382183908045977, Runtime: 143.01365 seconds
2026/04/07 06:50:45 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 3, Current best: 0.9382183908045977, Global best: 0.9382183908045977, Runtime: 147.41712 seconds
2026/04/07 06:53:09 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 4, Current best: 0.9382183908045977, Global best: 0.9382183908045977, Runtime: 144.04662 seconds
2026/04/07 06:55:42 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 5, Current best: 0.9388532477947074, Global best: 0.9388532477947074, Runtime: 153.28589 seconds
2026/04/07 06:58:10 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Proble

Best params: {'n_estimators': 211, 'max_depth': 29, 'min_samples_split': 2}
Best CV accuracy: 0.9407


2026/04/07 08:45:35 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: OriginalCSA(epoch=50, pop_size=30, p_a=0.3)


...

Best params: {'max_depth': 26, 'min_samples_split': 2}
Best CV accuracy: 0.9226


2026/04/07 08:53:56 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: OriginalCSA(epoch=50, pop_size=30, p_a=0.3)


Test accuracy: 0.9346
              precision    recall  f1-score   support

      Normal       0.98      0.98      0.98      1684
     TCP-SYN       0.99      0.98      0.98      1123
    PortScan       1.00      1.00      1.00       755
    Overflow       0.86      0.78      0.82       204
   Blackhole       0.89      0.91      0.90      1900
   Diversion       0.89      0.88      0.88      1816

    accuracy                           0.93      7482
   macro avg       0.94      0.92      0.93      7482
weighted avg       0.93      0.93      0.93      7482

Model saved to ../results/DT_CSA_model.pkl

Optimizing AB with CSA...


2026/04/07 08:57:16 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 1, Current best: 0.7954089815557337, Global best: 0.7954089815557337, Runtime: 112.18910 seconds
2026/04/07 08:59:12 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 2, Current best: 0.7954089815557337, Global best: 0.7954089815557337, Runtime: 116.08925 seconds
2026/04/07 09:01:02 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 3, Current best: 0.7954089815557337, Global best: 0.7954089815557337, Runtime: 109.50849 seconds
2026/04/07 09:02:58 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 4, Current best: 0.7954089815557337, Global best: 0.7954089815557337, Runtime: 116.08143 seconds
2026/04/07 09:05:04 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Problem: P, Epoch: 5, Current best: 0.7954089815557337, Global best: 0.7954089815557337, Runtime: 126.24010 seconds
2026/04/07 09:07:14 PM, INFO, mealpy.swarm_based.CSA.OriginalCSA: >>>Proble

Best params: {'n_estimators': 247, 'learning_rate': 0.8391456784442851}
Best CV accuracy: 0.7968
Test accuracy: 0.8038
              precision    recall  f1-score   support

      Normal       0.96      0.77      0.86      1684
     TCP-SYN       0.76      0.86      0.81      1123
    PortScan       1.00      1.00      1.00       755
    Overflow       0.38      0.71      0.50       204
   Blackhole       0.84      0.70      0.76      1900
   Diversion       0.71      0.83      0.77      1816

    accuracy                           0.80      7482
   macro avg       0.78      0.81      0.78      7482
weighted avg       0.83      0.80      0.81      7482

Model saved to ../results/AB_CSA_model.pkl


In [ ]:
# Confusion matrices — replicate paper's Fig. 3
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

for idx, (name, model) in enumerate(trained_models.items()):
    ax = axes.flatten()[idx]
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{name}-CSA (Acc: {results[name]['accuracy']:.2f})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

axes.flatten()[-1].axis("off")  # hide 6th subplot
plt.suptitle("Confusion Matrices — CSA-Optimized Classifiers on UNR-IDD", fontsize=14)
plt.tight_layout()
plt.savefig("../figures/confusion_matrices_all.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Comparison with original paper — fill in paper values manually
paper_accuracy = {
    "KNN-CSA": 0.79,
    "LR-CSA": 0.50,
    "RF-CSA": 0.75,
    "DT-CSA": 0.87,
    "AB-CSA": 0.88,
}

comparison = pd.DataFrame({
    "Model": [f"{n}-CSA" for n in trained_models.keys()],
    "Paper Accuracy": [paper_accuracy[f"{n}-CSA"] for n in trained_models.keys()],
    "Replicated Accuracy": [results[n]["accuracy"] for n in trained_models.keys()],
})
comparison["Difference"] = comparison["Replicated Accuracy"] - comparison["Paper Accuracy"]
print(comparison.to_string(index=False))

In [ ]:
# Save preprocessed data for use in XAI notebook
joblib.dump({
    "X_train": X_train, "X_test": X_test,
    "y_train": y_train, "y_test": y_test,
    "feature_names": feature_names, "scaler": scaler,
}, "../results/preprocessed_data.pkl")
print("Preprocessed data saved.")